In [1]:
import os
os.chdir(r"C:\Users\OWNER\Desktop\RAG\memorial_rag+sql") ## burası documentsin konumu
print("Çalışma dizini:", os.getcwd())

Çalışma dizini: C:\Users\OWNER\Desktop\RAG\memorial_rag+sql


In [2]:
DOCS_DIR = "documents"
CHUNKS_FILE = "chunks.jsonl"
CHROMA_DIR = "vector_db"
COLLECTION_NAME = "belgelerim"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

EMBEDDING_MODEL = "BAAI/bge-m3"         
TOP_K = 5

OLLAMA_MODEL = "qwen3:8b"
OLLAMA_HOST = "http://localhost:11434"

RAG_PROMPT_TEMPLATE = """Aşağıda verilen BAĞLAM bilgisini kullanarak kullanıcının SORUSUNU yanıtla.

Kurallar:
- Cevabını öncelikle BAĞLAM'daki bilgilere dayandır.
- BAĞLAM'da doğrudan yazmasa bile, verilen bilgilerden mantıklı ve açık bir çıkarım yapılabiliyorsa bu çıkarımı yap ve hangi bilgiye dayandığını belirt.
- Sadece BAĞLAM'da hiçbir ilgili bilgi yoksa "Bu bilgi elimdeki belgelerde yok" de.
- BAĞLAM'da yer almayan sayı, isim, tarih gibi somut bilgileri kesinlikle uydurma.

BAĞLAM:
{context}

SORU:
{question}

CEVAP:"""

In [3]:
import fitz  # PyMuPDF
import json
import pandas as pd

def parse_pdf(filepath):   # pdfi texte çevirir
    text_parts = []
    with fitz.open(filepath) as doc:
        for page in doc:
            text = page.get_text()
            if text.strip():
                text_parts.append(text)
    return "\n".join(text_parts)

def parse_txt(filepath):   # txt dosyasını okur
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def hucre_metni(deger):   #Excel hücresini metne çevirir.
    try:
        if pd.isna(deger):
            return None
    except (TypeError, ValueError):
        pass  # liste/dizi gibi tuhaf değerlerde pd.isna patlayabilir, metne çevirmeyi dene
    if isinstance(deger, pd.Timestamp):
        return deger.strftime("%d.%m.%Y")
    metin = str(deger).strip()
    return metin or None

def parse_excel(filepath):   # exceli chunka çevirir.
    """
    Dönen her eleman: {"text": embed edilecek metin, "sheet": sayfa adı, "row": satır no}
    Dosya adı / sayfa / satır bilgisi bilinçli olarak "text" içine YAZILMAZ;
    embedding'e gürültü katmasın diye metadata olarak taşınır.
    """
    satir_kayitlari = []
    sayfalar = pd.read_excel(filepath, sheet_name=None)  # None = tüm sayfaları oku

    for sayfa_adi, df in sayfalar.items():
        df = df.dropna(how="all")  # tamamen boş satırları at
        for satir_no, (_, satir) in enumerate(df.iterrows(), start=1):
            alanlar = []
            for sutun in df.columns:
                deger = hucre_metni(satir[sutun])
                if deger is not None:
                    alanlar.append(f"{sutun}: {deger}")
            if not alanlar:
                continue
            satir_kayitlari.append({
                "text": "\n".join(alanlar),
                "sheet": str(sayfa_adi),
                "row": satir_no,
            })

    return satir_kayitlari

def parse_documents(docs_dir=DOCS_DIR):
    parsed_docs = []
    if not os.path.isdir(docs_dir):
        os.makedirs(docs_dir, exist_ok=True)
        print(f"UYARI: '{docs_dir}' klasörü yoktu, oluşturuldu. Belgelerini koyup tekrar çalıştır.")
        return parsed_docs

    for filename in sorted(os.listdir(docs_dir)):
        filepath = os.path.join(docs_dir, filename)
        ext = filename.lower().split(".")[-1]
        try:
            # Excel: satır bazlı, "rows" anahtarıyla döner (metin splitter'ına girmez)
            if ext in ("xlsx", "xlsm", "xls"):
                if filename.startswith("~$"):
                    continue  # Excel'in açık dosya için oluşturduğu geçici kopya
                rows = parse_excel(filepath)
                if rows:
                    parsed_docs.append({"source": filename, "rows": rows})
                    print(f"Ayrıştırıldı: {filename} ({len(rows)} satır kaydı)")
                else:
                    print(f"UYARI: '{filename}' içinden satır çıkarılamadı.")
                continue

            # PDF ve TXT: düz metin, "text" anahtarıyla döner
            if ext == "pdf":
                text = parse_pdf(filepath)
            elif ext == "txt":
                text = parse_txt(filepath)
            else:
                print(f"Atlandı (desteklenmeyen format): {filename}")
                continue

            if text.strip():
                parsed_docs.append({"source": filename, "text": text})
                print(f"Ayrıştırıldı: {filename} ({len(text)} karakter)")
            else:
                print(f"UYARI: '{filename}' içinden metin çıkarılamadı.")
        except Exception as e:
            print(f"HATA ({filename}): {e}")
    return parsed_docs

In [4]:
parsed_docs = parse_documents()  # yukardaki parse fonksyonlarını tetikliyor
print(f"\nToplam {len(parsed_docs)} belge ayrıştırıldı.")
# parsed docs içinde tüm veriler string olarak sıralı kocaman bi satır halinde duruyo
#parsed docs metadata şeklinde source ve texti var

Ayrıştırıldı: doktorlar_v2.txt (20327 karakter)
Ayrıştırıldı: doktorlar_v2.xlsx (20 satır kaydı)
Ayrıştırıldı: hastalar_v2.txt (14680 karakter)
Ayrıştırıldı: hastalar_v2.xlsx (20 satır kaydı)
Ayrıştırıldı: subeler.xlsx (11 satır kaydı)

Toplam 5 belge ayrıştırıldı.


In [5]:
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import re

def structural_split(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    )
    return splitter.split_text(text)


# satırın tamamı -, =, _, *, ~ karakterlerinden (en az 3 tane) oluşuyorsa eşleşir
AYIRICI_SATIR = re.compile(r"^[ \t]*[-=_*~]{3,}[ \t]*$")


def ayiricilari_temizle(text):
    """Metinden ayırıcı satırları ('----', '====') tamamen atar.
    Embedding'e gürültü olarak girmesinler diye chunk kaydedilmeden önce çağrılır."""
    satirlar = [s for s in text.splitlines() if not AYIRICI_SATIR.match(s)]
    return "\n".join(satirlar).strip()


def ayirici_ile_split(text, min_uzunluk=20, maks_uzunluk=CHUNK_SIZE * 2):

    bloklar = []
    aktif = []

    for satir in text.splitlines():
        if AYIRICI_SATIR.match(satir):
            if aktif:                              # ayırıcıya geldik -> blok bitti
                bloklar.append("\n".join(aktif))
                aktif = []
        else:
            aktif.append(satir)

    if aktif:                                      # dosya ayırıcıyla bitmemişse son blok
        bloklar.append("\n".join(aktif))

    bloklar = [b.strip() for b in bloklar]
    bloklar = [b for b in bloklar if len(b) >= min_uzunluk]

    # Ayırıcı yoksa / tek blok çıktıysa bu yöntemin anlamı kalmaz
    if len(bloklar) <= 1:
        return structural_split(ayiricilari_temizle(text))

    parcalar = []
    for blok in bloklar:
        if len(blok) > maks_uzunluk:
            print(f"UYARI: {len(blok)} karakterlik blok tek chunk olarak eklendi "
                  f"(embedding seyrelebilir). İlk satır: {blok.splitlines()[0][:60]}")
        parcalar.append(blok)
    return parcalar

In [6]:
print(f"Embedding modeli yükleniyor: {EMBEDDING_MODEL} (ilk seferde ~2.2 GB indirir)...")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print("Model hazır.")

Embedding modeli yükleniyor: BAAI/bge-m3 (ilk seferde ~2.2 GB indirir)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model hazır.


In [7]:
# bolme -> bu chunk'ı hangi mekanizma üretti:
#   "satir"   Excel'de bir satır          -> tam kayıt
#   "blok"    txt'de iki ayırıcı arası    -> tam kayıt
#   "pencere" structural_split çıktısı    -> karakterle kesilmiş, ortadan bölünmüş olabilir
# Dosya formatı bilgisi zaten "source" içinde (.txt/.xlsx/.pdf), burada tekrarlanmıyor.

all_chunks = []
for doc in parsed_docs:
    source = doc["source"]

    if "rows" in doc:
        # Excel: her satır zaten bir chunk.
        # parcalar -> (metin, o parçaya ait ek metadata) çiftleri
        parcalar = []
        for row in doc["rows"]:
            ek_meta = {"sheet": row["sheet"], "row": row["row"]}
            if len(row["text"]) > CHUNK_SIZE * 2:
                parcalar.extend((p, ek_meta) for p in structural_split(row["text"]))
            else:
                parcalar.append((row["text"], ek_meta))
        bolme = "satir"
    else:
        # TXT -> dosyadaki '----' ayırıcılarına göre kayıt bazlı böl
        # PDF -> karakter bazlı yöntem
        if source.lower().endswith(".txt"):
            ham_parcalar = ayirici_ile_split(doc["text"])
            bolme = "blok"
        else:
            ham_parcalar = structural_split(doc["text"])
            bolme = "pencere"

        # Vektörleştirmeden önce artakalan ayırıcı satırları sil (gürültü olmasın)
        parcalar = [(ayiricilari_temizle(p), {}) for p in ham_parcalar]
        parcalar = [(p, m) for p, m in parcalar if p]   # temizlikten sonra boşalanları at

    print(f"{source}: {len(parcalar)} parça ({bolme})")

    for idx, (chunk_text, ek_meta) in enumerate(parcalar):
        all_chunks.append({
            "id": f"{source}::{idx}",
            "source": source,
            "chunk_index": idx,
            "bolme": bolme,
            "text": chunk_text,
            **ek_meta,
        })

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"\nToplam {len(all_chunks)} chunk oluşturuldu.")

doktorlar_v2.txt: 21 parça (blok)
doktorlar_v2.xlsx: 20 parça (satir)
hastalar_v2.txt: 22 parça (blok)
hastalar_v2.xlsx: 20 parça (satir)
subeler.xlsx: 11 parça (satir)

Toplam 94 chunk oluşturuldu.


In [8]:
import chromadb

def chunk_metadata(c):       #Chunk dict'inden Chroma metadata'sı üretir.

    meta = {                   
        "source": c["source"],
        "chunk_index": c["chunk_index"],
        "bolme": c["bolme"],
    }
    for anahtar in ("sheet", "row"):
        if anahtar in c:
            meta[anahtar] = c[anahtar]
    return meta

texts = [c["text"] for c in all_chunks]
print(f"{len(texts)} chunk embedding'e çevriliyor...")
chunk_embeddings = embed_model.encode(
    texts, normalize_embeddings=True, show_progress_bar=True, batch_size=16
)

client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
 #verinin Chroma'ya kaydedildiği yer
collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=chunk_embeddings.tolist(),
    documents=texts,
    metadatas=[chunk_metadata(c) for c in all_chunks],
)

print(f"Tamamlandı: {len(all_chunks)} chunk Chroma'ya kaydedildi.")

94 chunk embedding'e çevriliyor...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Tamamlandı: 94 chunk Chroma'ya kaydedildi.
